# 📗 LightRAG: 가벼운 그래프 색인과 키워드 검색

교안 01의 Microsoft GraphRAG는 색인 때 커뮤니티마다 커뮤니티 보고서를 LLM으로 써 두어 LLM 호출이 많습니다. **LightRAG**는 커뮤니티 보고서를 쓰지 않아 색인이 가볍습니다. 이 교안은 교안 01과 같은 **곡성·구례 관광지 안내문 32편**과 골드 질문(정답 근거 문서를 미리 정해 둔 평가용 질문)으로 LightRAG를 실행합니다.

**학습 목표**

1. LightRAG가 Microsoft GraphRAG와 무엇이 다른지 설명합니다.
2. LightRAG로 색인하고 검색 모드 다섯 개를 비교합니다.
3. 색인 결과 그래프를 열어 보고, 문서 한 편을 추가·삭제하며 LLM 호출 수를 잽니다.
4. 답변 모델에 넘긴 청크가 어느 문서에서 왔는지 찾아, 정답 근거 문서를 넘겼는지 확인합니다.

## 1. LightRAG와 실무 쓰임

LightRAG(홍콩대학교, 2024)도 색인 때 LLM으로 개체·관계를 뽑아 지식 그래프를 만듭니다. Microsoft GraphRAG와 다른 점은 셋입니다.

- **커뮤니티 보고서가 없습니다.** 개체를 커뮤니티로 묶어 요약문을 쓰는 단계가 없어 색인이 가볍습니다.
- **관계 이름 대신 키워드를 붙입니다.** 관계 이름(종류)은 뽑지 않고, 관계마다 설명과 "등산로, 연결" 같은 키워드를 붙입니다. 질문의 주제 키워드와 비슷한 관계 설명·키워드를 벡터 검색해 관계를 찾습니다.
- **그래프 DB 없이 파일에 저장합니다.** 기본 설정은 그래프, 벡터(개체·관계 설명과 원문 청크), 원문을 폴더 안 파일에 둡니다. 필요하면 Neo4j 같은 그래프 DB로 바꿀 수 있습니다.

**앞 단원의 도메인 GraphRAG와 다른 점**

| | 도메인 GraphRAG(앞 단원) | LightRAG |
|---|---|---|
| 그래프 | 사람이 노드·관계 종류를 설계하고 적재 | LLM이 문서에서 개체·관계를 뽑음. 관계는 종류 이름 없이 설명과 키워드 |
| 검색 | 에이전트가 질문마다 Cypher를 작성 | 키워드로 개체·관계를 벡터 검색하고 이웃을 정해진 방식으로 가져옴(Cypher 없음) |
| 잘 맞는 질문 | 관계가 정해진 정확한 조회 | 정리되지 않은 문서에서 관련 내용 모으기 |

<img src="images/02_lightrag_overview.png" width="1050" alt="Microsoft GraphRAG는 개체·관계를 추출한 뒤 커뮤니티로 묶고 묶음마다 커뮤니티 보고서를 씁니다. LightRAG는 개체·관계를 추출해 지식 그래프와 벡터로 저장하며 커뮤니티 보고서를 쓰지 않습니다. 새 문서가 오면 그 문서만 추출해 기존 그래프에 합칩니다.">

**실무에서 쓰는 이유**

- **색인이 가볍습니다.** 32편 색인에 LLM을 LightRAG는 34회, GraphRAG는 147회 불렀습니다(강사 측정).
- **문서를 추가하고 지울 수 있습니다.** 새 문서는 그 문서만 추출해 합치고(`ainsert`), 뺄 문서는 `adelete_by_doc_id`로 지웁니다. 고친 문서는 지운 뒤 다시 넣습니다.
- **기능이 계속 추가됩니다.** Microsoft `graphrag`는 저장소 README에 유지보수 모드(새 기능 없이 버그·의존성 수정만)라고 밝혔고, `lightrag-hku`는 새 기능을 계속 더하고 있습니다.
- **설치가 간단합니다.** 기본은 파일 저장이라 DB 없이 시작하고, 운영에서는 그래프를 Neo4j·PostgreSQL, 벡터를 Milvus·Qdrant 등으로 바꿉니다.

**생각해 보기**: LightRAG에는 커뮤니티 보고서가 없습니다. 자료 전체에 흩어진 항목을 모두 꼽는 전역 질문에 GraphRAG Global처럼 답할 수 있을까요?

<details><summary>정답 보기</summary>

일부만 답합니다. 키워드로 찾은 개체·관계 설명과 원문 청크 5개(이 교안 설정)만 넘기므로, 정답 근거 문서가 이 5개 안에 다 들지 않으면 나머지 문서에만 있는 항목은 빠집니다. 검색 모드 비교에서 G1의 청크 문서를 정답 근거 문서와 견주어 확인합니다.

</details>

## 2. 실습 준비

### 2-1. 라이브러리와 모델

교안 01과 같은 가상환경과 모델을 쓰고, LightRAG 패키지만 새로 씁니다. 모델이 같으므로 결과 차이는 검색 방식에서 나옵니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

# 노트북이 있는 단원 폴더의 data와 output을 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 노트북 폴더부터 위로 올라가며 처음 찾은 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 키는 저장소 최상위 .env에서 읽습니다(이미 읽은 키는 유지).
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델(.env에 없으면 gpt-5.6-luna)
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")

print("답변 모델:", model_name)

### 2-2. 관광지 안내 자료와 골드 질문

관광지 안내문 33편(한국관광공사 TourAPI, 공공누리 제1유형) 가운데 32편을 색인하고, 남은 1편은 문서 추가 시연에 씁니다. 골드 질문 12개는 단일 사실 S1~S4, 멀티홉(여러 문서를 이어 읽어야 답이 나옴) M1~M5, 전역(자료 전체에 흩어진 내용을 모아야 함) G1~G3입니다.

아래 셀은 문서 목록과 골드 질문을 읽고, 교안 01과 같은 평가 함수 `evidence_hit`·`make_result_row`를 정의합니다. 결과 표 한 행의 `hit_docs`/`gold_docs`가 근거 문서 포함률(정답 근거 문서 가운데 답변 모델에 넘긴 문서의 비율)입니다.

In [ ]:
def read_json(name):
    """data 폴더 아래 name 경로의 JSON 파일을 읽어 list 또는 dict로 반환합니다.

    name: data 폴더 기준 상대 경로(예: "본실습/questions.json")
    """
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


# 문서 추가 시연용 1편은 색인할 문서 목록에서 빼야 하므로, 그 문서 ID를 여기서 먼저 읽습니다.
extra_id = read_json("본실습/extra.json")["doc_id"]
doc_paths = []

for path in sorted((data_dir / "본실습" / "docs").glob("*.txt")):
    # path.stem은 확장자를 뺀 파일 이름(문서 ID)입니다.
    if path.stem != extra_id:
        doc_paths.append(path)

# 골드 질문마다 id, question, qtype, gold_doc_ids(정답 근거 문서 ID)가 있습니다.
questions = read_json("본실습/questions.json")
# 질문 id로 찾도록 dict로 만듭니다.
questions_by_id = {question["id"]: question for question in questions}

TOP_K = 5  # 답변 모델에 넘길 청크 수
# 답 형식 지시: 전역 질문은 항목을 모두 나열하게, 나머지는 짧게 답하게 합니다.
SHORT_ANSWER = "한국어로 한두 문장. 근거에 없으면 모른다고 답한다"
LIST_ANSWER = "한국어로, 해당하는 항목을 모두 나열한 목록"


def evidence_hit(given_ids, gold_ids):
    """정답 근거 문서 가운데 검색이 넘긴 문서가 몇 편인지 세어 int로 반환합니다.

    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록
    gold_ids: 질문의 정답 근거 문서 ID 목록(골드 질문의 gold_doc_ids)
    예: 정답 근거 3편 가운데 2편을 넘겼으면 2를 반환합니다.
    """
    hit = 0

    # 정답 근거 문서마다 넘긴 목록에 들어 있는지 봅니다.
    for gold_id in gold_ids:
        if gold_id in given_ids:
            hit += 1

    return hit


def make_result_row(method, question, given_ids, usage, seconds):
    """검색 한 번의 결과를 비교용 결과 표의 한 행(dict)으로 만들어 반환합니다.

    method: 검색 방식 이름(예: "graphrag_local", "lightrag_mix")
    question: 골드 질문 dict(id, qtype, gold_doc_ids)
    given_ids: 검색이 답변 모델에 넘긴 원문 문서 ID 목록. 원문을 넘기지 않는 GraphRAG Global은 None을 넘기고, 이때 문서 열은 빈칸(None)이 됩니다.
    usage: 이 검색의 LLM 사용량 dict(llm_calls 호출 수, input_tokens 입력 토큰, output_tokens 출력 토큰)
    seconds: 검색에 걸린 시간(초)
    문서 열: docs_given(넘긴 문서 수), gold_docs(정답 근거 수), hit_docs(정답 근거 가운데 넘긴 수), all_hit(모두 넘겼으면 1, 아니면 0)
    """
    # 사용량과 시간은 모든 방식에 채우고, 문서 열은 빈칸(None)으로 시작합니다.
    row = {"method": method, "question_id": question["id"], "qtype": question["qtype"],
           "docs_given": None, "gold_docs": None, "hit_docs": None, "all_hit": None,
           "llm_calls": usage["llm_calls"], "input_tokens": usage["input_tokens"],
           "output_tokens": usage["output_tokens"], "seconds": round(seconds, 1)}

    # 원문 문서를 넘긴 방식만 문서 열을 채웁니다.
    if given_ids is not None:
        gold_ids = question["gold_doc_ids"]
        hit = evidence_hit(given_ids, gold_ids)
        row["docs_given"] = len(given_ids)
        row["gold_docs"] = len(gold_ids)
        row["hit_docs"] = hit

        # 정답 근거를 모두 넘겼으면 1
        if hit == len(gold_ids):
            row["all_hit"] = 1
        else:
            row["all_hit"] = 0

    return row


print("색인할 문서:", len(doc_paths), "편 | 추가할 문서 ID:", extra_id, "| 골드 질문:", len(questions), "개")

## 3. LightRAG

### 3-1. 구체 키워드로 개체, 주제 키워드로 관계 검색

LightRAG의 검색 모드는 다섯 가지입니다. `naive`를 뺀 네 모드는 먼저 LLM이 질문에서 **구체 키워드**(low-level, 예: `섬진강`, `철새`)와 **주제 키워드**(high-level, 예: `생태 관찰`)를 뽑고, 그 키워드로 개체나 관계를 벡터 검색한 뒤 그래프에서 이어진 것을 함께 가져옵니다. 그래프를 따라가는 방식이 코드에 정해져 있어 Cypher를 쓰지 않습니다.

**키워드는 어떻게 뽑나요?** 사람이 정하지 않습니다. 질문마다 LLM을 한 번 불러, LightRAG에 내장된 키워드 추출 프롬프트로 두 목록을 JSON으로 받습니다.

- **주제 키워드:** 질문의 의도와 주제(예: `조선 시대 선비`, `유학 교육 장소`)
- **구체 키워드:** 질문에 나온 고유명사와 구체적인 항목(예: `곡성`, `구례`, `유학`)

프롬프트는 질문에 있는 말로만 뽑게 하고, `addon_params`의 언어(여기서는 한국어)로 쓰게 합니다. LLM이 뽑으므로 같은 질문도 실행마다 키워드가 조금씩 달라지고, 그만큼 검색 결과도 달라질 수 있습니다. 키워드를 고정하려면 `QueryParam(hl_keywords=..., ll_keywords=...)`로 직접 넘깁니다. 아래 검색 모드 비교에서 이 방법을 씁니다.

<img src="images/02_lightrag_dual_level.png" width="1050" alt="LightRAG는 질문에서 구체 키워드와 주제 키워드를 뽑아 각각 개체 검색과 관계 검색에 쓰고(hybrid), 질문 임베딩으로 찾은 청크까지 더해(mix) 답변에 넘길 문맥을 만듭니다.">

그림의 벡터 저장소는 세 곳입니다. 개체 벡터(개체 이름 + 설명), 관계 벡터(관계 키워드 + 양 끝 개체 + 설명), 청크 벡터(원문 청크)이고, 모드마다 쓰는 곳이 다릅니다. 키워드는 목록을 쉼표로 이어 한 문장으로 임베딩합니다.

**`naive`: 원문 청크만 찾는 벡터 검색**

1. 질문을 그대로 임베딩합니다. 키워드는 뽑지 않습니다.
2. 청크 벡터에서 질문과 가까운 청크 `chunk_top_k`개를 찾습니다.
3. 청크만 답변 모델에 넘깁니다. 개체·관계는 넘기지 않습니다.

**`local`: 구체 키워드로 개체를 찾고 이어진 관계를 가져옴**

1. LLM이 키워드를 뽑고(1회), 구체 키워드를 임베딩합니다.
2. 개체 벡터에서 가까운 개체 `top_k`개(기본 40)를 찾습니다.
3. 그래프에서 그 개체에 이어진 관계를 연결이 많고 강도가 큰 순으로 가져옵니다. 원문 청크 후보는 찾은 개체가 나온 청크 가운데 질문 임베딩과 가까운 것입니다.

**`global`: 주제 키워드로 관계를 찾고 양 끝 개체를 가져옴**

1. LLM이 키워드를 뽑고(1회), 주제 키워드를 임베딩합니다.
2. 관계 벡터에서 가까운 관계 `top_k`개(기본 40)를 찾습니다. 관계마다 "등산로, 연결" 같은 키워드가 붙어 있어 `탐방 코스` 같은 주제어가 관계와 잘 맞습니다.
3. 그 관계의 양 끝 개체를 가져옵니다. 원문 청크 후보는 가져온 양 끝 개체(와 관계)가 나온 청크 가운데 질문 임베딩과 가까운 것입니다.

LightRAG의 `global`은 관계를 벡터 검색하는 모드입니다. 커뮤니티 보고서를 읽는 교안 01의 GraphRAG Global과는 이름만 같습니다.

**`hybrid`: `local`과 `global`을 함께**

두 검색을 모두 하고 개체·관계·청크 후보를 합쳐 겹친 것을 뺍니다. BM25 키워드 검색과 벡터 검색을 섞는 하이브리드 검색과는 다른 뜻입니다.

**`mix`: `hybrid` + 질문 임베딩으로 찾은 청크**

`hybrid`에 더해 `naive`처럼 청크 벡터에서 질문과 가까운 청크 `chunk_top_k`개를 찾아 청크 후보에 넣습니다. `QueryParam`의 기본 모드입니다.

**모든 모드의 마지막 두 단계**

1. 청크 후보를 갈래별로 번갈아 합치며 겹친 것을 빼고, 리랭커가 연결되어 있으면 다시 줄 세운 뒤 `chunk_top_k`개(기본 20, 이 교안 5)로 자릅니다.
2. 개체·관계·청크를 한 문맥으로 답변 모델에 넘깁니다(LLM 1회). 그래서 질문 하나에 LLM은 `naive` 1회, 나머지 모드 2회입니다.

<img src="images/02_lightrag_modes.png" width="1050" alt="LightRAG 검색 모드와 잘 맞는 질문: 관광안내소 직원이 관광 안내 챗봇에 문의하는 네 장면입니다. naive(단일 사실)는 침실습지 생물 수를 묻는 질문에 비슷한 원문 조각을 찾고, local(특정 개체)은 화엄사계곡 길이 어디로 이어지는지에 그 개체와 이어진 관계를 찾습니다. global(주제)은 선비가 공부하던 곳을 모두 묻는 질문에 주제가 맞는 관계를 찾고, mix(멀티홉)는 황전마을에서 노고단고개까지 걸리는 시간에 개체·관계와 원문 조각을 함께 찾습니다.">

| 모드 | 벡터 검색하는 곳 | 잘 맞는 질문 | 약점 |
|---|---|---|---|
| `naive` | 청크 | 단일 사실(S1: 침실습지에서 확인된 생물 수) | 그래프를 쓰지 않아 여러 문서를 잇는 질문에 약함 |
| `local` | 개체 | 특정 개체 주변(예: 화엄사계곡 길이 이어지는 곳) | 질문에 고유명사가 없으면 출발할 개체를 잘못 찾음 |
| `global` | 관계 | 주제로 묶어 묻는 질문(G1: 선비들이 공부하던 곳) | 전체를 미리 요약해 두지 않아, 찾은 관계와 청크 5개 밖의 항목은 빠짐 |
| `hybrid` | 개체 + 관계 | 개체와 주제가 함께 있는 질문 | 두 갈래 청크가 `chunk_top_k` 자리를 나눠 가짐 |
| `mix` | 개체 + 관계 + 청크 | 멀티홉(M1: 황전마을에서 노고단고개까지 걸리는 시간). `QueryParam`의 기본 모드 | 세 갈래 청크가 `chunk_top_k` 자리를 나눠 가짐 |

### 3-2. 색인: 모델 함수 연결과 한국어 설정

LightRAG는 LLM과 임베딩 모델을 **함수**로 받습니다. 아래 셀은 두 모델 함수와 LLM 호출 계수기 `tracker`를 만듭니다.

In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc, TokenTracker

# 임베딩 모델: 벡터를 768차원으로 줄여 씁니다(교안 01과 같은 설정).
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768)
# LightRAG 색인 파일을 저장할 폴더
lightrag_dir = output_dir / "lightrag"
# LLM 호출 수와 토큰 수를 더해 두는 계수기. 응답이 올 때마다 호출 함수가 사용량을 더합니다.
tracker = TokenTracker()


async def course_llm(prompt, **kwargs):
    """prompt를 답변 모델(model_name)에 보내 응답 글(str)을 반환하고, 호출 수와 토큰 수를 tracker에 더합니다.

    LightRAG가 개체·관계 추출, 설명 요약, 질문 키워드 추출, 답변 작성에 부르는 LLM 함수입니다.
    prompt: LightRAG가 만든 프롬프트 글
    kwargs: LightRAG가 함께 넘기는 system_prompt 등. 그대로 openai_complete_if_cache에 전달합니다.
    """
    # openai_complete_if_cache는 LightRAG가 제공하는 OpenAI 호출 함수입니다.
    return await openai_complete_if_cache(model_name, prompt, token_tracker=tracker, **kwargs)


async def course_embed(texts):
    """texts의 글을 한 번에 임베딩해 (글 수, 768) 모양의 numpy 배열로 반환합니다.

    LightRAG가 청크·개체·관계를 벡터 저장소에 넣을 때와 질문 키워드로 검색할 때 부릅니다.
    texts: 임베딩할 글(str) 목록
    """
    # LangChain 임베딩은 벡터 리스트를 돌려주므로 LightRAG가 받는 numpy 배열로 바꿉니다.
    return np.array(await embedding_model.aembed_documents(texts))


# 벡터 차원(768)을 함께 알려 주도록 EmbeddingFunc로 감쌉니다.
embedding_func = EmbeddingFunc(embedding_dim=768, func=course_embed)

32편을 색인하고 LLM 호출 수를 잽니다. LightRAG는 추출·요약·답변에 라이브러리에 내장된 프롬프트를 쓰고, 이 교안은 `addon_params`로 그 프롬프트에 들어갈 언어와 개체 유형 안내만 바꿉니다. 색인·검색 함수는 비동기라 `await`로 부릅니다.

- **다시 색인하려면 커널을 다시 시작합니다.** 같은 커널에서 이 셀을 다시 실행하면, 폴더를 지워도 메모리에 남은 기록 때문에 이미 넣은 문서로 보고 호출이 0회입니다.
- **기다리기 어렵다면** 아래 색인 셀 대신 그다음 셀을 실행해, 강사가 같은 설정으로 미리 만든 색인(`data/본실습/prebuilt/`)을 복사합니다. 색인은 10분 안팎 걸리고 LLM을 약 35회 부릅니다. 두 셀 가운데 하나만 실행합니다.

In [ ]:
import shutil
import time

from lightrag import LightRAG

# 이전 tour 색인을 지웁니다. 남아 있으면 이미 넣은 문서로 보고 추출하지 않습니다.
# shutil.rmtree는 폴더를 안의 파일까지 지웁니다. ignore_errors=True면 폴더가 없어도 오류가 나지 않습니다.
shutil.rmtree(lightrag_dir / "tour", ignore_errors=True)
rag = LightRAG(
    working_dir=str(lightrag_dir),  # 색인 파일을 저장할 폴더
    workspace="tour",  # working_dir 안의 하위 폴더. 따라하기 색인(incident)과 저장소를 나눕니다.
    llm_model_func=course_llm,  # 개체 추출, 설명 요약, 답변에 쓸 LLM 함수
    embedding_func=embedding_func,  # 개체, 관계, 청크를 벡터로 바꿀 함수
    # 설명을 한국어로 쓰고(기본 English), 개체를 관광 자료에 맞는 유형으로 분류하게 합니다.
    addon_params={"language": "Korean",
                  "entity_types_guidance": "장소, 인물, 단체, 행사 중 하나로 분류한다. 맞는 것이 없으면 Other."},
    entity_extract_max_gleaning=0,  # 추출을 한 번 더 시키는 단계(기본 1회)를 끕니다. 교안 01과 같은 조건
    # 검색(키워드 추출·답변) 응답 캐시를 꺼 질문마다 LLM 호출을 셉니다.
    # 추출 응답 캐시는 따로 켜져 있어, 문서를 지울 때 남은 문서의 추출 결과로 설명을 다시 만드는 데 씁니다.
    enable_llm_cache=False,
)

# 그래프·벡터·문서 상태 저장소를 엽니다. 색인과 검색 전에 한 번 부릅니다.
await rag.initialize_storages()

# 문서 한 편이 텍스트 하나인 목록으로 넘깁니다.
texts = [path.read_text(encoding="utf-8") for path in doc_paths]
# 파일 이름을 함께 넘기면 검색 결과의 청크가 어느 문서에서 왔는지 알 수 있습니다.
file_paths = [path.name for path in doc_paths]
# 계수기를 0으로 되돌리고 색인 호출만 셉니다.
tracker.reset()
start = time.time()
# 청크마다 추출 1회, 설명이 많이 쌓인 개체와 관계는 요약 1회를 더 부릅니다.
await rag.ainsert(texts, file_paths=file_paths)
seconds = time.time() - start

print("LightRAG 색인 호출:", tracker.call_count, "회 | 입력 토큰:", tracker.prompt_tokens, "| 출력 토큰:", tracker.completion_tokens, "| 걸린 시간:", round(seconds), "초")

In [ ]:
import shutil

from lightrag import LightRAG

# 위 색인 셀 대신 실행합니다. 강사가 같은 설정으로 만든 tour 색인을 lightrag_dir/tour에 복사합니다.
# shutil.copytree는 폴더를 하위 파일까지 복사합니다. dirs_exist_ok=True면 이미 있는 폴더에 덮어씁니다.
shutil.copytree(data_dir / "본실습" / "prebuilt" / "lightrag_tour", lightrag_dir / "tour", dirs_exist_ok=True)

# 색인 셀과 같은 인자로 만듭니다. 저장소를 열면 복사한 파일을 읽으므로 문서는 넣지 않습니다.
rag = LightRAG(
    working_dir=str(lightrag_dir),
    workspace="tour",
    llm_model_func=course_llm,
    embedding_func=embedding_func,
    addon_params={"language": "Korean",
                  "entity_types_guidance": "장소, 인물, 단체, 행사 중 하나로 분류한다. 맞는 것이 없으면 Other."},
    entity_extract_max_gleaning=0,
    enable_llm_cache=False,
)
await rag.initialize_storages()

# 강사 색인의 호출 수·토큰·시간
index_usage = read_json("본실습/prebuilt/index_usage.json")["lightrag"]

print("강사 색인 호출:", index_usage["llm_calls"], "회 | 입력 토큰:", index_usage["input_tokens"], "| 출력 토큰:", index_usage["output_tokens"], "| 걸린 시간:", round(index_usage["seconds"]), "초")

### 3-3. 색인 결과 보기

LightRAG는 그래프 DB 없이, 색인을 `working_dir/workspace` 폴더의 파일로 저장합니다(기본 저장소).

| 저장하는 것 | 파일 | 기본 저장소 | 검색 때 쓰이는 방법 |
|---|---|---|---|
| 지식 그래프(개체·관계) | `graph_chunk_entity_relation.graphml` | `NetworkXStorage` | 찾은 개체·관계에 이어진 이웃을 연결로 가져옴 |
| 개체 벡터 | `vdb_entities.json` | `NanoVectorDBStorage` | `local`·`hybrid`·`mix`: 구체 키워드 임베딩과 개체(이름 + 설명) 임베딩을 비교 |
| 관계 벡터 | `vdb_relationships.json` | `NanoVectorDBStorage` | `global`·`hybrid`·`mix`: 주제 키워드 임베딩과 관계(키워드 + 양 끝 개체 + 설명) 임베딩을 비교 |
| 청크 벡터 | `vdb_chunks.json` | `NanoVectorDBStorage` | `naive`·`mix`: 질문 임베딩과 청크 임베딩을 비교. 나머지 모드도 찾은 개체·관계의 청크를 고를 때 씀 |
| 원문·청크·문서 처리 상태 | `kv_store_*.json` | `JsonKVStorage` 등 | 찾은 청크의 원문 글을 꺼내고, 이미 넣은 문서인지 확인함 |

**추출한 것을 검색에서 어떻게 쓰나요?** LightRAG도 BM25 같은 키워드 일치 검색은 하지 않습니다. LLM이 뽑은 키워드조차 임베딩해서, 벡터 저장소 세 곳에서 의미가 가까운 것을 찾습니다(임베딩 유사도 검색). 찾은 개체·관계에서는 그래프 파일의 연결을 따라 이웃과 원문 청크를 함께 가져옵니다.

그래프 DB로 바꿀 때는 `LightRAG(graph_storage="Neo4JStorage", ...)`처럼 저장소 이름을 바꾸고, 접속 정보(`NEO4J_URI` 등)를 환경 변수로 줍니다.

**관계 이름 대신 무엇이 있나요?** 추출 프롬프트가 관계 하나를 (개체 1, 개체 2, 관계 키워드, 설명) 네 값으로만 받아 관계 이름이 없습니다. 관계 키워드도 색인 때 LLM이 문서를 읽고 붙이며, 프롬프트는 관계를 요약하는 상위 키워드를 한 개 이상 쉼표로 쓰게 합니다.

아래 셀은 그래프 파일을 읽어 노고단 주변 관계의 키워드와 설명을 봅니다. 결과를 볼 때 두 가지를 기억합니다.

- **`source`와 `target`은 방향이 아닙니다.** 관계는 방향 없이 저장되고, 저장 순서만 다릅니다. 예를 들어 `노고단, 오산(구례)` 관계의 설명은 "오산(구례) 정상은 지리산 노고단을 볼 수 있는 조망점이다."입니다.
- **키워드는 정해진 목록이 아닙니다.** `연결`, `위치` 같은 일반어가 많고 표기도 제각각이라, Cypher의 관계 종류처럼 정확히 거르는 데는 못 씁니다. 대신 질문의 주제 키워드와 의미가 가까운 관계를 벡터 검색으로 찾는 데 씁니다.

In [ ]:
import networkx as nx


def read_graph(workspace):
    """LightRAG가 저장한 지식 그래프 파일(graphml)을 읽어 NetworkX 그래프로 반환합니다.

    workspace: 색인할 때 LightRAG에 준 workspace 이름(예: "tour"). lightrag_dir/<workspace>/graph_chunk_entity_relation.graphml 파일을 읽습니다.
    """
    return nx.read_graphml(lightrag_dir / workspace / "graph_chunk_entity_relation.graphml")


# 노드 이름이 개체 이름이고, 관계(edge) 속성에 keywords와 description이 있습니다.
graph = read_graph("tour")

print("개체:", graph.number_of_nodes(), "개 | 관계:", graph.number_of_edges(), "개")

# 이름에 노고단이 들어간 개체(노고단고개 등)에 이어진 관계만 표로 봅니다.
relation_rows = []

for source, target, edge in graph.edges(data=True):
    if "노고단" in source or "노고단" in target:
        relation_rows.append({"source": source, "target": target,
                              "keywords": edge["keywords"], "description": edge["description"]})

display(pd.DataFrame(relation_rows))

### 3-4. `aquery_data`로 검색 모드 비교

`aquery_data`는 검색만 합니다. 답변은 만들지 않고 **답변 모델에 넘길 개체·관계·청크만** 돌려주며, LLM은 키워드 추출에 한 번만 부릅니다. 그래서 답변 비용 없이 모드마다 무엇을 찾는지 비교하기 쉽습니다.

답변까지 받으려면 같은 `rag`의 `aquery_llm`을 부릅니다. 색인 때 넘긴 `llm_model_func`(`course_llm`)가 답변도 쓰므로 LLM을 따로 붙이지 않습니다. 아래 답변 생성에서 씁니다.

**리랭킹은 끄고 검색합니다.** 리랭킹(reranking)은 검색으로 찾은 원문 청크의 순서를 리랭커로 다시 매기는 단계입니다.

- **리랭커:** 질문과 청크를 한 입력으로 함께 읽어 관련도 점수를 내는 모델(Cross-Encoder)입니다. 임베딩 검색은 질문과 청크를 따로 벡터로 바꿔 비교해 빠르지만 덜 정확하고, 리랭커는 둘을 함께 읽어 더 정확한 대신 느립니다. 그래서 임베딩으로 후보를 넉넉히 찾고, 리랭커로 다시 줄 세워 앞의 몇 개만 넘깁니다.
- **LightRAG에서의 위치:** 원문 청크만 리랭킹하고, `chunk_top_k`개로 자르기 전에 적용합니다. 개체·관계는 리랭킹하지 않습니다.
- **LightRAG에는 리랭커가 내장되어 있지 않습니다.** `LightRAG(rerank_model_func=...)`로 직접 연결합니다. `lightrag.rerank`에 Cohere, Jina, 알리바바 리랭킹 API를 부르는 함수가 있고, 로컬 모델을 부르는 함수를 직접 만들어 넣을 수도 있습니다.
- **`enable_rerank=True`만으로는 리랭킹이 일어나지 않습니다.** 기본값이 True지만, 리랭커를 연결하지 않으면 경고만 남기고 원래 순서를 그대로 씁니다.
- 본문은 리랭커 없이 GraphRAG와 같은 조건으로 비교하려고 `enable_rerank=False`로 끕니다. 리랭커를 연결한 검색은 함께 따라하기에서 해 봅니다.

아래 셀은 먼저 G1 질문의 키워드를 한 번 뽑아 둡니다.

In [ ]:
from lightrag import QueryParam


def chunk_doc_ids(data):
    """aquery_data 검색 결과에서 답변 모델에 넘길 원문 청크가 나온 문서 ID를 중복 없이 정렬한 list로 반환합니다.

    data: rag.aquery_data(...)가 반환한 dict. 개체·관계는 LLM이 쓴 설명이라 세지 않고 원문 청크만 봅니다. 청크가 없으면 빈 list를 반환합니다.
    """
    doc_ids = set()

    # data["data"]["chunks"]는 답변 모델에 넘길 원문 청크 목록이고, 청크마다 file_path가 있습니다.
    for chunk in data["data"]["chunks"]:
        # 색인 때 넘긴 "<문서 ID>.txt"에서 확장자를 떼면 문서 ID입니다.
        doc_ids.add(chunk["file_path"].removesuffix(".txt"))

    return sorted(doc_ids)


demo_question = questions_by_id["G1"]

print("질문", demo_question["id"] + ":", demo_question["question"])

# 키워드를 얻으려고 mix 모드로 한 번 검색합니다.
# enable_rerank=False: 리랭커를 연결하지 않았으므로 리랭킹을 끕니다(켜 두면 경고만 남깁니다).
keyword_data = await rag.aquery_data(demo_question["question"], param=QueryParam(mode="mix", enable_rerank=False))
# 검색 결과의 metadata에 LLM이 뽑은 두 종류의 키워드가 들어 있습니다.
keywords = keyword_data["metadata"]["keywords"]

print("주제 키워드(high-level):", keywords["high_level"], "| 구체 키워드(low-level):", keywords["low_level"])
print("G1 정답 근거 문서:", demo_question["gold_doc_ids"])

다섯 모드를 같은 키워드로 검색해 찾은 개체·관계·청크 수를 비교합니다.

In [ ]:
mode_rows = []

for mode in ["naive", "local", "global", "hybrid", "mix"]:
    # chunk_top_k: 원문 청크를 다른 방식과 같은 5개까지만 받습니다(기본 20개).
    # hl_keywords, ll_keywords: 뽑아 둔 키워드를 넘기면 키워드 추출 호출을 하지 않습니다.
    param = QueryParam(mode=mode, chunk_top_k=TOP_K, enable_rerank=False,
                       hl_keywords=keywords["high_level"], ll_keywords=keywords["low_level"])
    data = await rag.aquery_data(demo_question["question"], param=param)
    # data["data"]에 답변 모델에 넘길 entities, relationships, chunks가 들어 있습니다.
    mode_rows.append({"모드": mode, "개체": len(data["data"]["entities"]),
                      "관계": len(data["data"]["relationships"]), "청크": len(data["data"]["chunks"]),
                      "청크 문서": chunk_doc_ids(data)})

display(pd.DataFrame(mode_rows))

**결과 읽는 법**

세 열은 답변 모델에 넘길 문맥에 든 개수입니다. **개체**는 개체 이름과 설명, **관계**는 두 개체와 관계 키워드·설명, **청크**는 원문 조각입니다.

| 모드 | 개체·관계 | 청크 후보를 모으는 곳 |
|---|---|---|
| `naive` | 그래프를 쓰지 않아 0 | 질문 임베딩으로 찾은 청크 |
| `local` | 구체 키워드로 개체를 `top_k`(기본 40)개 찾고, 그 개체에 이어진 관계가 따라옴 | 찾은 개체가 나온 청크 |
| `global` | 주제 키워드로 관계를 40개 찾고, 양 끝 개체가 따라와 개체가 관계보다 많을 수 있음 | 양 끝 개체(와 관계)가 나온 청크 |
| `hybrid` | `local`과 `global` 결과를 합치고 겹친 것을 빼서 두 모드의 합보다 조금 적음 | 위 두 곳 |
| `mix` | `hybrid`와 거의 같음(경계의 한두 개는 다를 수 있음) | 위 두 곳 + 질문 임베딩으로 찾은 청크 |

- **청크가 모두 5개인 이유:** 후보를 모으는 곳은 모드마다 다르지만, 어느 모드든 후보가 5개보다 많고 `chunk_top_k=5`로 넘길 개수를 5개로 잘랐기 때문입니다. 그래서 모드가 달라도 청크 문서가 크게 다르지 않을 수 있습니다.
- **청크 문서를 G1 정답 근거 문서와 견주어 봅니다.** 정답 근거가 4편이라 5개 안에 다 들어가야 원문 근거를 모두 넘긴 것입니다.
- **개체·관계 설명도 답변 모델에 넘어갑니다.** 청크 5개에 들지 못한 문서의 내용도 설명으로 일부 전해지지만, LLM이 요약한 글이라 원문 근거로는 세지 않습니다.

### 3-5. 답변 생성과 근거 확인

`aquery_llm`은 `aquery_data`와 같은 검색 결과로 답변까지 만듭니다. 아래 셀은 멀티홉 질문 M1을 `mix` 모드로 검색·답변하고 결과 표 한 행을 만듭니다.

- **M1의 정답 근거:** 황전마을 안내문(황전 계곡이 화엄사계곡의 지류)과 화엄사계곡 안내문(노고단고개까지 걸리는 시간)이 함께 넘어가야 답할 수 있습니다.
- **근거 문서는 원문 청크의 문서만 셉니다.** 함께 넘기는 개체·관계에도 `file_path`가 있지만, 그 설명은 여러 청크를 합친 LLM 요약이라 원문 근거로 세지 않습니다.

질문 여러 개는 함께 따라하기에서 실행하고, 방식 비교는 부록에서 합니다.

In [ ]:
import time

question = questions_by_id["M1"]

print("질문", question["id"] + ":", question["question"])

# response_type: 답 형식 지시(멀티홉 질문이라 짧은 답)
param = QueryParam(mode="mix", chunk_top_k=TOP_K, enable_rerank=False, response_type=SHORT_ANSWER)
# 계수기를 0으로 되돌려 이 질문의 사용량만 셉니다.
tracker.reset()
start = time.time()
# 검색과 답변을 한 번에 합니다. LLM은 키워드 추출 1회, 답변 1회입니다.
result = await rag.aquery_llm(question["question"], param=param)
seconds = time.time() - start

# tracker 값을 make_result_row가 받는 usage 형식으로 옮깁니다.
usage = {"llm_calls": tracker.call_count, "input_tokens": tracker.prompt_tokens,
         "output_tokens": tracker.completion_tokens}

# aquery_llm 결과에도 aquery_data와 같은 data가 있어 chunk_doc_ids를 그대로 씁니다.
given_ids = chunk_doc_ids(result)

print("넘긴 문서 ID:", given_ids, "| 정답 근거 문서:", question["gold_doc_ids"])
display(pd.DataFrame([make_result_row("lightrag_mix", question, given_ids, usage, seconds)]))

M1 답변을 출력합니다. 답변 끝 `### References`의 파일 이름으로 답변이 어느 원문을 근거로 삼았는지 대조합니다.

In [ ]:
from IPython.display import Markdown

# 결과에는 답변(llm_response)과 넘긴 개체, 관계, 청크(data)가 함께 들어 있습니다.
display(Markdown(result["llm_response"]["content"]))

### 3-6. 문서 추가와 삭제

새 안내문 1편을 같은 색인에 추가하고 LLM 호출 수를 잽니다. LightRAG는 새 문서만 추출하고, 이름이 같은 개체는 기존 노드에 합칩니다. 나중에 지울 수 있게 문서 ID(`ids`)를 정해 넣습니다.

In [ ]:
# 색인에서 빼 두었던 문서(extra_id)의 파일 경로
extra_path = data_dir / "본실습" / "docs" / f"{extra_id}.txt"
# 추가 전 개체 이름을 모아 둡니다.
before_entities = set(read_graph("tour").nodes)
tracker.reset()
# 한 편이므로 목록 대신 텍스트, 문서 ID, 파일 이름을 그대로 넘깁니다.
# ids를 넘기지 않으면 LightRAG가 파일 이름을 해시해 "doc-"로 시작하는 문서 ID를 만듭니다.
await rag.ainsert(extra_path.read_text(encoding="utf-8"), ids=extra_id, file_paths=extra_path.name)

print("추가 문서:", extra_path.name, "| 갱신 호출:", tracker.call_count, "회")

# 이 문서에서 처음 생긴 개체
new_entities = set(read_graph("tour").nodes) - before_entities

print("새로 생긴 개체:", sorted(new_entities))

`adelete_by_doc_id`는 문서와 그 청크를 지우고, 그 문서에서만 나온 개체·관계도 지웁니다. 다른 문서에도 나온 개체는 남은 문서의 추출 결과(추출 응답 캐시)로 설명을 다시 만듭니다.

**내용을 고친 문서는 지운 뒤 다시 넣습니다.** 문서 ID나 파일 이름이 같은 문서를 다시 넣기만 하면 이미 있는 문서로 보고 건너뜁니다.

In [ ]:
tracker.reset()
# 문서 ID로 지웁니다. 설명이 많이 쌓인 개체를 다시 요약할 때만 LLM을 부릅니다.
delete_result = await rag.adelete_by_doc_id(extra_id)

# status가 success면 삭제된 것입니다.
print("삭제 결과:", delete_result.status, "| 삭제 호출:", tracker.call_count, "회")

after_entities = set(read_graph("tour").nodes)

# 추가 문서에서만 나온 개체가 남았는지, 기존 문서에도 있던 당몰샘은 남았는지 봅니다.
print("남은 새 개체:", sorted(new_entities & after_entities))
print("당몰샘 개체:", [name for name in after_entities if "당몰샘" in name])

# 메모리에 있는 그래프와 벡터를 파일에 저장하고 저장소를 닫습니다.
await rag.finalize_storages()

### 3-7. 색인 그래프 시각화

LightRAG가 저장한 그래프 파일에서 노고단 주변만 골라 pyvis로 그려 봅니다. pyvis는 NetworkX처럼 노드와 관계를 넣으면 마우스로 끌고 확대할 수 있는 HTML 그래프를 만들어 주는 라이브러리입니다.

- **선 위의 글자:** 관계 키워드입니다. LightRAG는 관계 이름 대신 이 키워드를 붙입니다.
- **마우스를 올리면** 개체·관계의 설명이 보입니다.

그래프는 `output/lightrag_graph.html`로도 저장합니다.

In [ ]:
import html

from IPython.display import HTML
from pyvis.network import Network


def show_network(net, html_path):
    """pyvis 그래프 net을 html_path에 HTML 파일로 저장하고, 노트북 출력 칸에 띄웁니다.

    net: 노드와 관계를 넣은 pyvis Network
    html_path: 저장할 HTML 파일 경로. 이 파일을 브라우저로 열어도 같은 그래프가 나옵니다.
    """
    net.write_html(str(html_path))
    page = html_path.read_text(encoding="utf-8")

    # HTML 문서를 iframe 안에 넣어 노트북 출력 칸에 그대로 띄웁니다.
    display(HTML(f'<div><iframe srcdoc="{html.escape(page)}" width="100%" height="680" style="border:0"></iframe></div>'))


graph = read_graph("tour")
# 이름에 노고단이 들어간 개체와, 그 개체에 바로 이어진 개체만 고릅니다.
nogodan_names = {name for name in graph.nodes if "노고단" in name}
keep_names = set(nogodan_names)

for name in nogodan_names:
    keep_names |= set(graph.neighbors(name))

# subgraph: 고른 노드와 그 노드끼리의 관계만 남긴 그래프
nogodan_graph = graph.subgraph(keep_names)

# height·width: 그림 크기. cdn_resources="in_line"은 그리기 스크립트를 HTML 안에 넣어 인터넷 없이도 열리게 합니다.
lightrag_net = Network(height="650px", width="100%", cdn_resources="in_line")

for name, node in nogodan_graph.nodes(data=True):
    # title은 마우스를 올리면 보이는 글이고, group(개체 유형)이 같으면 같은 색입니다.
    lightrag_net.add_node(name, label=name, title=node["description"], group=node["entity_type"])

for source, target, edge in nogodan_graph.edges(data=True):
    # 선 위에 관계 키워드를, 마우스 글로 관계 설명을 붙입니다.
    lightrag_net.add_edge(source, target, label=edge["keywords"], title=edge["description"])

print("노고단 주변 개체:", nogodan_graph.number_of_nodes(), "개 | 관계:", nogodan_graph.number_of_edges(), "개")

show_network(lightrag_net, output_dir / "lightrag_graph.html")

## 4. 🖐️ 함께 따라하기: 장애 지식베이스에서 되풀이된 장애 원인 찾기

### 4-1. 질문과 작성 목표

**누리페이**(수업용 가상 핀테크 회사)의 장애 지식베이스 20편으로 같은 과정을 반복합니다. 분기 장애 회고에서 "4월부터 9월까지 누리페이 장애 보고서에서 되풀이해 나타난 원인 유형은 무엇인가?"(Q7)를 묻는 상황입니다.

**작성 목표**

1. 20편을 색인하고, 골드 질문 8개마다 Jev가 고른 모드로 검색·답변합니다. 결과는 `output/practice_results_lesson02.csv`에 저장합니다.
2. 리랭커를 연결해 멀티홉 질문의 근거 문서가 달라지는지 봅니다.
3. 새 장애 보고서(`INC-0922`)를 추가했다가 회수 요청에 따라 지웁니다.

**힌트:** 본문의 색인, 답변 생성, 문서 추가·삭제 코드에서 workspace(`incident`)와 개체 유형만 바꿉니다. 변수 이름은 `practice_`로 시작하고, 본문에서 정의한 함수와 `tracker`는 그대로 씁니다. Jev 사용법은 교안 01 함께 따라하기와 같고, 키(`TYPESAFE_API_KEY`)는 `.env`에서 읽습니다.

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **직접 설계:** 지시 블록에 적힌 이름·입력·반환값에 맞춰 코드 전체를 작성합니다.
- **부분 작성:** 코드의 `...` 자리만 채웁니다. 이미 적힌 경로·출력 코드는 그대로 실행합니다.
- `...`가 남은 채로 실행하면 오류가 나거나 빈 결과가 나옵니다. 작성 셀을 완성한 뒤 바로 아래 셀을 실행해 결과를 봅니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 직접 작성해 봅니다.

### 4-2. 장애 보고서와 골드 질문 읽기

색인할 20편, 나중에 추가할 장애 보고서 한 편, 골드 질문 8개를 읽습니다.

In [ ]:
practice_doc_paths = sorted((data_dir / "따라하기" / "docs").glob("*.txt"))
# 색인 뒤에 추가할 새 장애 보고서
practice_extra_path = data_dir / "따라하기" / "extra" / "INC-0922.txt"
practice_questions = read_json("따라하기/questions.json")
# 질문 id로 찾도록 dict로 만듭니다.
practice_questions_by_id = {question["id"]: question for question in practice_questions}

print("장애 지식베이스:", len(practice_doc_paths), "편 | 추가 장애 보고서:", practice_extra_path.name, "| 골드 질문:", len(practice_questions), "개")
print("Q7:", practice_questions_by_id["Q7"]["question"], "| 정답 근거 문서 수:", len(practice_questions_by_id["Q7"]["gold_doc_ids"]), "편")

### 4-3. LightRAG 색인, 골드 질문 검색, 리랭킹 비교, 새 장애 보고서 추가와 삭제

**리랭커 준비.** 리랭킹 단원에서 쓴 로컬 리랭커 `Qwen/Qwen3-Reranker-0.6B`를 이 지식베이스 색인에 연결합니다.

- **LightRAG가 리랭커를 부르는 방식:** 리랭커 함수를 `(질문, 청크 글 목록, top_n)`으로 부르고, 돌려받은 `{"index": 청크 번호, "relevance_score": 점수}` 목록의 순서대로 청크를 다시 줄 세웁니다.
- **색인 전에 준비합니다.** 리랭커 함수는 `LightRAG(rerank_model_func=...)`로 **만들 때** 넘깁니다. 만든 뒤에 속성을 바꾸면 그래프를 쓰는 모드(`mix` 등)에는 적용되지 않습니다.
- 모델을 불러올 때 `HF_TOKEN`, 진행 막대(`TqdmWarning`), `Default prompt name` 경고가 나올 수 있는데 무시해도 됩니다.

**부분 작성 · 리랭커 함수**

**작성:** `practice_rerank`가 `cross_encoder.score`로 (질문, 청크) 쌍마다 점수를 매기고, 점수가 높은 순서로 `{"index", "relevance_score"}` 목록을 반환하도록 채웁니다.

**반환값:** 다음 색인 셀에서 `LightRAG`에 넘길 `practice_rerank` 함수입니다. 셀 끝의 확인 줄에서 관련 있는 글이 0번으로 나와야 합니다.

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 첫 실행에는 공개 모델 파일(약 1.2GB)을 내려받습니다. CPU와 입력 상한 1024토큰을 씁니다.
cross_encoder = HuggingFaceCrossEncoder(
    model_name="Qwen/Qwen3-Reranker-0.6B",
    model_kwargs={"device": "cpu", "max_length": 1024},
)


async def practice_rerank(query, documents, top_n=None, **kwargs):
    """리랭커로 documents를 query와의 관련도 순으로 다시 매겨 LightRAG가 받는 형식의 list로 반환합니다.

    query: 질문 글
    documents: LightRAG가 넘기는 원문 청크 글 목록
    top_n: 남길 개수. LightRAG가 chunk_top_k 값을 넘깁니다. None이면 모두 남깁니다.
    kwargs: LightRAG가 함께 넘길 수 있는 나머지 인자(쓰지 않음)
    반환: [{"index": documents 안의 번호, "relevance_score": 점수}, ...] 점수가 높은 순
    """
    # 1. (질문, 청크) 쌍 목록을 cross_encoder.score에 넘겨 관련도 점수를 매기세요.
    scores = ...
    # 점수가 높은 청크 번호부터 줄 세웁니다.
    order = sorted(range(len(documents)), key=lambda index: scores[index], reverse=True)

    if top_n:
        order = order[:top_n]

    # 2. order의 번호마다 {"index": 번호, "relevance_score": 점수} dict를 만들어 목록으로 반환하세요.
    return ...


# 리랭커 동작 확인: 질문과 관련 있는 첫 글(0번)이 1위로 나와야 합니다.
print("리랭커 확인:", await practice_rerank("결제 승인 지연 원인", ["결제 승인 서버 장애 보고서", "사내 체육대회 안내"], top_n=1))

**직접 설계 · 장애 지식베이스 색인**

**작성:**

1. `lightrag_dir / "incident"` 폴더를 지우고 `workspace="incident"`인 `practice_rag`를 만든 뒤 저장소를 엽니다.
   - 개체 유형 안내: "서비스, 구성요소, 팀, 인물, 장애 중 하나로 분류한다. 맞는 것이 없으면 Other."
   - 리랭커: `rerank_model_func=practice_rerank`
   - 나머지 인자는 본문의 관광지 색인과 같습니다.
2. `practice_doc_paths` 20편의 텍스트와 파일 이름 목록을 만듭니다.
3. `tracker`를 0으로 되돌리고 `ainsert`로 넣은 뒤 호출 수·토큰·시간을 출력합니다.

**반환값:** 색인 호출 수, 토큰 수, 걸린 시간을 출력합니다.

같은 커널에서 이 셀을 다시 돌리면 이미 넣은 문서로 보고 건너뛰므로, 다시 색인하려면 커널을 재시작합니다.

In [ ]:
# 색인을 기다리기 어려우면 이 셀 대신 바로 아래 셀로 강사가 미리 만든 색인을 복사합니다. 둘 중 하나만 실행합니다.
# 1. 이전 incident 색인 폴더를 지우고 practice_rag를 만들어 저장소를 여세요.
#    workspace="incident", 개체 유형(서비스, 구성요소, 팀, 인물, 장애) 안내, rerank_model_func=practice_rerank를 넣고 나머지 인자는 본문 색인과 같습니다.
# 2. practice_doc_paths 20편의 텍스트와 파일 이름 목록을 만드세요.
# 3. tracker를 0으로 되돌리고 ainsert로 넣은 뒤 호출 수·토큰·시간을 출력하세요.
...

색인은 5분 안팎 걸리고 LLM을 약 20회 부르므로, 기다리기 어렵다면 위 색인 셀 대신 아래 셀로 강사가 미리 만든 색인(`data/따라하기/prebuilt/`)을 복사합니다. 두 셀 가운데 하나만 실행하고, 같은 커널에서 색인 셀을 이미 실행했다면 이 셀은 건너뜁니다.

In [ ]:
# 위 장애 지식베이스 색인 셀 대신 실행합니다. 같은 커널에서 색인 셀을 이미 실행했다면 건너뜁니다.
# 강사가 같은 설정으로 만든 incident 색인을 lightrag_dir/incident에 복사합니다.
shutil.copytree(data_dir / "따라하기" / "prebuilt" / "lightrag_incident", lightrag_dir / "incident", dirs_exist_ok=True)

# 색인 셀과 같은 인자로 만듭니다. 저장소를 열면 복사한 파일을 읽으므로 문서는 넣지 않습니다.
practice_rag = LightRAG(
    working_dir=str(lightrag_dir),
    workspace="incident",
    llm_model_func=course_llm,
    embedding_func=embedding_func,
    addon_params={"language": "Korean",
                  "entity_types_guidance": "서비스, 구성요소, 팀, 인물, 장애 중 하나로 분류한다. 맞는 것이 없으면 Other."},
    entity_extract_max_gleaning=0,
    enable_llm_cache=False,
    # 리랭커 함수. 검색에서 enable_rerank=True일 때만 씁니다.
    rerank_model_func=practice_rerank,
)
await practice_rag.initialize_storages()

# 강사 색인의 호출 수·토큰·시간
practice_index_usage = read_json("따라하기/prebuilt/index_usage.json")["lightrag"]

print("강사 색인 호출:", practice_index_usage["llm_calls"], "회 | 입력 토큰:", practice_index_usage["input_tokens"], "| 출력 토큰:", practice_index_usage["output_tokens"], "| 걸린 시간:", round(practice_index_usage["seconds"]), "초")

교안 01처럼 질문 유형을 Jev로 먼저 판단하고 검색 모드를 고릅니다. 기준은 모드 비교 표의 "잘 맞는 질문"을 따르되, 자료 전체를 모아 정리하는 전역 질문은 원문 청크까지 찾는 `mix`로 보냅니다. `naive`는 키워드를 뽑지 않아 답변 호출 1회로 끝나고, 나머지 모드는 키워드 추출과 답변으로 2회를 부릅니다.

Jev가 고른 모드는 골드 질문의 유형과 다를 수 있습니다. 1위 확률이 낮고 2위와 비슷한 질문(예: 0.4 대 0.37)은 경계에 있어 실행마다 바뀔 수 있으므로, 결과 표에서 Jev 선택과 골드 유형을 함께 봅니다.

**부분 작성 · Jev 검색 모드 분류기**

**작성:** `practice_router`를 `TypeSafeClassifier`로 만듭니다. `model`은 환경변수 `TYPESAFE_MODEL`을 쓰고, 없으면 `"jev-1.13.0"`입니다. 네 모드의 기준(`practice_route_question`)은 제공합니다.

**반환값:** 출력에 Jev 모델 이름과 네 모드(`naive`, `local`, `global`, `mix`)가 나옵니다.

In [ ]:
from langchain_typesafe import Choice, TypeSafeClassifier

# 1. practice_router를 TypeSafeClassifier로 만드세요. model은 환경변수 TYPESAFE_MODEL을 쓰고, 없으면 "jev-1.13.0"입니다.
practice_router = ...

# 질문마다 네 모드 가운데 하나를 고르는 기준입니다. 답을 얻기까지 몇 단계를 거치는지로 나눕니다.
practice_route_question = Choice(
    instructions="질문에 맞는 LightRAG 검색 모드를 고르세요. 답을 얻기까지 거쳐야 하는 대상이 몇 단계인지, 자료 전체를 모아야 하는지 먼저 보세요.",
    criteria={
        "naive": "단일 사실: 한 문서에 적힌 수치·시간 같은 사실 하나를 묻고, 다른 대상으로 이어 갈 필요가 없는 질문",
        "local": "특정 개체 주변: 개체 하나를 정해 그 개체와 바로 이어진 정보를 두루 묻는 질문",
        "global": "주제로 묶어 묻는 질문: 특정 개체 없이 한 주제에 해당하는 대상을 찾는 질문",
        "mix": "멀티홉·전역: 한 문서의 답을 다른 문서의 대상으로 이어서 찾아야 하는, 연결을 두 단계 이상 따라가는 질문이거나, 자료 전체에 흩어진 내용을 모아 목록·유형·경향을 정리하는 질문",
    },
)

print("Jev 모델:", practice_router.model, "| 검색 모드:", list(practice_route_question.criteria))

**부분 작성 · Jev가 고른 모드로 골드 질문 검색과 결과 표 저장**

**작성:**

1. `practice_router.invoke`에 `{"state": 질문 글, "questions": {"route": practice_route_question}}`를 넘겨 `route_decision`에 담습니다.
2. `route_decision.choices["route"].choice`를 `route`에 담습니다.
3. 본문 M1 검색과 같은 조건(청크 `TOP_K`개, 리랭킹 끔, `response_type`)에서 모드만 `route`로 바꾼 `QueryParam`을 `param`에 담습니다.

**반환값:** 행 8개인 `practice_lightrag_rows`와 CSV 파일입니다. 줄마다 질문 유형(`qtype`)과 Jev의 선택이 함께 출력됩니다.

In [ ]:
practice_lightrag_rows = []

for question in practice_questions:
    # 전역 질문은 목록으로, 나머지는 짧게 답하게 합니다.
    if question["qtype"] == "global":
        response_type = LIST_ANSWER
    else:
        response_type = SHORT_ANSWER

    # 1. practice_router.invoke에 질문 글(state)과 {"route": practice_route_question}(questions)을 넘겨 route_decision에 담으세요.
    route_decision = ...
    # 2. route_decision.choices["route"]에서 Jev가 고른 모드(choice)를 route에 담으세요.
    route = ...
    # Jev가 고른 값이 네 모드 가운데 하나인지 확인합니다.
    assert route in practice_route_question.criteria, route

    # 3. 모드가 route이고 청크 TOP_K개, 리랭킹 끔, response_type을 넣은 QueryParam을 만드세요.
    param = ...
    # 질문마다 사용량을 따로 셉니다.
    tracker.reset()
    start = time.time()
    result = await practice_rag.aquery_llm(question["question"], param=param)
    seconds = time.time() - start
    usage = {"llm_calls": tracker.call_count, "input_tokens": tracker.prompt_tokens,
             "output_tokens": tracker.completion_tokens}
    given_ids = chunk_doc_ids(result)
    row = make_result_row("lightrag_" + route, question, given_ids, usage, seconds)
    # Jev의 선택과 그 선택의 확률을 함께 남깁니다.
    row["route"] = route
    row["route_probability"] = round(route_decision.choices["route"].probabilities[route], 2)
    practice_lightrag_rows.append(row)

    print(question["id"], question["qtype"], "| Jev 선택:", route, "| 넘긴 문서 ID:", given_ids)

# 결과 표를 CSV로 저장합니다.
pd.DataFrame(practice_lightrag_rows).to_csv(output_dir / "practice_results_lesson02.csv", index=False)

**리랭킹 전후 근거 비교.** 같은 색인에서 리랭킹만 켜고 꺼 답변 모델에 넘기는 원문 문서가 달라지는지 봅니다. 리랭커는 API가 아니라 내 컴퓨터에서 돌아 비용은 없지만, CPU에서는 질문 하나에 1분 안팎 걸립니다. 그래서 정답 근거가 3편인 멀티홉 질문 Q5, Q6만 비교합니다.

**부분 작성 · 리랭킹 전후 근거 비교**

**작성:** Q5, Q6마다 리랭킹을 끈 검색과 켠 검색의 `QueryParam`을 채웁니다. 켠 검색은 끈 검색에서 뽑은 키워드를 그대로 씁니다.

**반환값:** 질문마다 정답 근거 수와 리랭킹 전후의 정답 근거 포함 수를 담은 `practice_rerank_table`입니다.

In [ ]:
practice_rerank_rows = []

for question_id in ["Q5", "Q6"]:
    question = practice_questions_by_id[question_id]
    gold_ids = question["gold_doc_ids"]
    # 1. 리랭킹을 끈 mix 검색 조건(청크 TOP_K개)을 만드세요.
    param_off = ...
    data_off = await practice_rag.aquery_data(question["question"], param=param_off)
    # 첫 검색에서 뽑은 키워드를 그대로 써서 키워드 차이 없이 리랭킹 효과만 봅니다.
    keywords = data_off["metadata"]["keywords"]
    # 2. 같은 키워드(hl_keywords, ll_keywords)로 리랭킹을 켠 mix 검색 조건을 만드세요.
    param_on = ...
    data_on = await practice_rag.aquery_data(question["question"], param=param_on)

    given_off = chunk_doc_ids(data_off)
    given_on = chunk_doc_ids(data_on)

    print(question_id, "리랭킹 끔:", given_off, "| 켬:", given_on, "| 정답 근거:", gold_ids)

    practice_rerank_rows.append({"question_id": question_id, "gold_docs": len(gold_ids),
                                 "hit_docs_off": evidence_hit(given_off, gold_ids),
                                 "hit_docs_on": evidence_hit(given_on, gold_ids)})

practice_rerank_table = pd.DataFrame(practice_rerank_rows)

display(practice_rerank_table)

리랭킹은 후보를 새로 찾지 않고 이미 찾은 후보의 순서만 바꿉니다. 그래서 정답 근거가 후보 안에 있었지만 순위가 밀려 잘리던 경우에만 도움이 됩니다. 결과는 실행마다 다를 수 있으므로, 운영에서는 여러 질문으로 포함률과 걸린 시간을 함께 재 보고 켤지 정합니다.

**직접 설계 · 새 장애 보고서 추가**

**작성:**

1. 넣기 전 `read_graph("incident")`의 개체 이름을 집합 `practice_before_entities`에 담습니다.
2. `tracker`를 0으로 되돌리고, 검색을 마친 `practice_rag`에 `practice_extra_path`의 장애 보고서 한 편을 문서 ID `practice_extra_path.stem`(`INC-0922`)으로 넣은 뒤 갱신 호출 수를 출력합니다.
3. 새로 생긴 개체 이름을 집합 `practice_new_entities`에 담아 출력합니다.

**반환값:** `practice_new_entities`와 `갱신 호출:` 줄입니다.

In [ ]:
# 1. 넣기 전 incident 그래프의 개체 이름을 practice_before_entities에 담으세요.
# 2. 새 장애 보고서(practice_extra_path)를 문서 ID practice_extra_path.stem으로 넣고 갱신 호출 수를 출력하세요.
# 3. 새로 생긴 개체 이름을 practice_new_entities에 담아 출력하세요.
...

`INC-0922`는 근본 원인 검토가 끝나지 않은 초안이 잘못 올라간 것이라 회수 요청이 왔습니다. 초안이 남아 있으면 되풀이된 원인을 묻는 Q7 답에 확정되지 않은 원인이 섞이므로 지식베이스에서 지웁니다.

**부분 작성 · 잘못 올린 장애 보고서 지우기**

**작성:**

1. `adelete_by_doc_id`에 지울 장애 보고서의 문서 ID(`practice_extra_path.stem`)를 넘깁니다.
2. 삭제 뒤 `read_graph("incident")`의 개체 이름을 집합 `practice_after_entities`에 담습니다.

**반환값:** `삭제 결과:`, `남은 새 개체:` 줄입니다.

In [ ]:
tracker.reset()
# 문서 ID로 INC-0922 장애 보고서를 지웁니다.
practice_delete_result = await practice_rag.adelete_by_doc_id(...)

print("삭제 결과:", practice_delete_result.status, "| 삭제 호출:", tracker.call_count, "회")

# 삭제 뒤 incident 그래프의 개체 이름 집합
practice_after_entities = ...

# 이 장애 보고서에서만 나온 개체가 남았는지 봅니다.
print("남은 새 개체:", sorted(practice_new_entities & practice_after_entities))

# 메모리에 있는 그래프와 벡터를 파일에 저장하고 저장소를 닫습니다.
await practice_rag.finalize_storages()

## 5. 핵심 정리

- **LightRAG:** 색인 때 개체·관계를 뽑아 그래프와 벡터로 저장하고 커뮤니티 보고서는 쓰지 않아, 색인 호출이 GraphRAG보다 적습니다.
- **모드:** 구체 키워드로 개체 벡터를, 주제 키워드로 관계 벡터(관계 키워드 + 설명)를 검색합니다. `mix`는 여기에 질문 임베딩으로 찾은 청크를 더합니다.
- **색인 결과:** 그래프는 graphml 파일에 저장되며, 관계마다 키워드와 설명이 붙습니다.
- **문서 추가·삭제:** 새 문서만 추출해 합치므로 추가 호출이 색인 호출보다 훨씬 적습니다. `adelete_by_doc_id`로 지우면 그 문서에서만 나온 개체·관계도 사라집니다.
- **근거 확인:** 개체·관계 설명은 LLM 요약이라, 원문 근거는 청크의 파일 이름으로 확인합니다.
- **모드 고르기:** Jev가 질문 유형을 보고 모드를 고르게 하면, 단일 사실 질문은 키워드를 뽑지 않는 `naive`로 답변 호출 1회에 끝냅니다.